In [2]:
import pandas as pd
import numpy as np
import joblib

In [3]:
df = pd.read_csv("../../Datasets_For_Model_Training/Final_Merged_Dataset.csv")
df.head()

,Date,Electricity_Requirement,Humidity,Rainfall,Electricity_Supply,Solar_Irradiance,Temperature
0,01-04-2015,8361.0,70.07,130.566857,8112.0,166.53,28.57
1,01-05-2015,8381.0,77.22,160.792286,8165.0,155.03,27.95
2,01-06-2015,8302.0,77.55,98.240286,8257.0,160.31,27.31
3,01-07-2015,8953.0,73.18,37.804286,8901.0,166.41,27.68
4,01-08-2015,8535.0,72.58,63.944286,8531.0,167.12,27.85


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Date                     129 non-null    object 
 1   Electricity_Requirement  129 non-null    float64
 2   Humidity                 129 non-null    float64
 3   Rainfall                 129 non-null    float64
 4   Electricity_Supply       129 non-null    float64
 5   Solar_Irradiance         129 non-null    float64
 6   Temperature              129 non-null    float64
dtypes: float64(6), object(1)
memory usage: 7.2+ KB


In [5]:
df["Date"] = pd.to_datetime(df["Date"])

In [6]:
# Convert datetime to string
df["Date"] = df["Date"].dt.strftime("%Y-%d-%m")

# Parse using the correct format (Year-Day-Month)
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%Y-%m-%d"
)

# Display
print(df["Date"].head())

0   2015-04-01
1   2015-05-01
2   2015-06-01
3   2015-07-01
4   2015-08-01
Name: Date, dtype: datetime64[ns]


In [7]:
# extracting year and month
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

In [8]:
import numpy as np

# Cyclical Encoding of Month
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

# Verify the encoding
print(
    df[["Month", "Month_sin", "Month_cos"]]
    .drop_duplicates()
    .sort_values("Month")
)

    Month     Month_sin     Month_cos
9       1  5.000000e-01  8.660254e-01
10      2  8.660254e-01  5.000000e-01
11      3  1.000000e+00  6.123234e-17
0       4  8.660254e-01 -5.000000e-01
1       5  5.000000e-01 -8.660254e-01
2       6  1.224647e-16 -1.000000e+00
3       7 -5.000000e-01 -8.660254e-01
4       8 -8.660254e-01 -5.000000e-01
5       9 -1.000000e+00 -1.836970e-16
6      10 -8.660254e-01  5.000000e-01
7      11 -5.000000e-01  8.660254e-01
8      12 -2.449294e-16  1.000000e+00


In [9]:
# Festival Feature Creation
# Pongal occurs in January (Month 1) every year.
# Deepavali changes between October (10) and November (11) depending on the year.
deepavali_months = {
    2015: 11,
    2016: 10,
    2017: 10,
    2018: 11,
    2019: 10,
    2020: 11,
    2021: 11,
    2022: 10,
    2023: 11,
    2024: 10,
    2025: 10
}

df["Festival"] = 0
df.loc[df["Month"] == 1, "Festival"] = 1

for year, festival_month in deepavali_months.items():
    df.loc[(df["Year"] == year) & (df["Month"] == festival_month), "Festival"] = 1


In [10]:
print(df["Date"].head(15))

0    2015-04-01
1    2015-05-01
2    2015-06-01
3    2015-07-01
4    2015-08-01
5    2015-09-01
6    2015-10-01
7    2015-11-01
8    2015-12-01
9    2016-01-01
10   2016-02-01
11   2016-03-01
12   2016-04-01
13   2016-05-01
14   2016-06-01
Name: Date, dtype: datetime64[ns]


In [11]:
df.dropna(inplace=True)

In [12]:
df.head()

,Date,Electricity_Requirement,Humidity,Rainfall,Electricity_Supply,Solar_Irradiance,Temperature,Year,Month,Month_sin,Month_cos,Festival
0,2015-04-01,8361.0,70.07,130.566857,8112.0,166.53,28.57,2015,4,8.660254e-01,-0.500000,0
1,2015-05-01,8381.0,77.22,160.792286,8165.0,155.03,27.95,2015,5,5.000000e-01,-0.866025,0
2,2015-06-01,8302.0,77.55,98.240286,8257.0,160.31,27.31,2015,6,1.224647e-16,-1.000000,0
3,2015-07-01,8953.0,73.18,37.804286,8901.0,166.41,27.68,2015,7,-5.000000e-01,-0.866025,0
4,2015-08-01,8535.0,72.58,63.944286,8531.0,167.12,27.85,2015,8,-8.660254e-01,-0.500000,0


In [13]:
df.shape

(129, 12)

In [14]:
df.isnull().sum()

Date                       0
Electricity_Requirement    0
Humidity                   0
Rainfall                   0
Electricity_Supply         0
Solar_Irradiance           0
Temperature                0
Year                       0
Month                      0
Month_sin                  0
Month_cos                  0
Festival                   0
dtype: int64

# Lag Feature Cell

In [15]:
# Create lag features for Electricity Requirement

df["Demand_Lag_1"] = df["Electricity_Requirement"].shift(1)
df["Demand_Lag_2"] = df["Electricity_Requirement"].shift(2)
df["Demand_Lag_3"] = df["Electricity_Requirement"].shift(3)


# Remove rows with missing lag values
df = df.dropna().reset_index(drop=True)

print(df.head())

        Date  Electricity_Requirement  Humidity    Rainfall  \
0 2015-07-01                   8953.0     73.18   37.804286   
1 2015-08-01                   8535.0     72.58   63.944286   
2 2015-09-01                   8498.0     75.12  119.769714   
3 2015-10-01                   8330.0     78.86  153.807429   
4 2015-11-01                   6511.0     87.29  316.494857   

   Electricity_Supply  Solar_Irradiance  Temperature  Year  Month  Month_sin  \
0              8901.0            166.41        27.68  2015      7  -0.500000   
1              8531.0            167.12        27.85  2015      8  -0.866025   
2              8373.0            157.25        27.50  2015      9  -1.000000   
3              8324.0            143.69        26.60  2015     10  -0.866025   
4              6508.0             92.62        25.04  2015     11  -0.500000   

      Month_cos  Festival  Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  
0 -8.660254e-01         0        8302.0        8381.0        8361.0  


In [16]:
print(df["Month"].unique())

[ 7  8  9 10 11 12  1  2  3  4  5  6]


In [17]:
print(df["Month"].value_counts().sort_index())

Month
1     10
2     10
3     10
4     10
5     10
6     10
7     11
8     11
9     11
10    11
11    11
12    11
Name: count, dtype: int64


In [18]:
print(df[["Date", "Month", "Festival"]].head(20))


         Date  Month  Festival
0  2015-07-01      7         0
1  2015-08-01      8         0
2  2015-09-01      9         0
3  2015-10-01     10         0
4  2015-11-01     11         1
5  2015-12-01     12         0
6  2016-01-01      1         1
7  2016-02-01      2         0
8  2016-03-01      3         0
9  2016-04-01      4         0
10 2016-05-01      5         0
11 2016-06-01      6         0
12 2016-07-01      7         0
13 2016-08-01      8         0
14 2016-09-01      9         0
15 2016-10-01     10         1
16 2016-11-01     11         0
17 2016-12-01     12         0
18 2017-01-01      1         1
19 2017-02-01      2         0


In [19]:
print(df.columns.tolist())

['Date', 'Electricity_Requirement', 'Humidity', 'Rainfall', 'Electricity_Supply', 'Solar_Irradiance', 'Temperature', 'Year', 'Month', 'Month_sin', 'Month_cos', 'Festival', 'Demand_Lag_1', 'Demand_Lag_2', 'Demand_Lag_3']


In [20]:
df.head()

,Date,Electricity_Requirement,Humidity,Rainfall,Electricity_Supply,Solar_Irradiance,Temperature,Year,Month,Month_sin,Month_cos,Festival,Demand_Lag_1,Demand_Lag_2,Demand_Lag_3
0,2015-07-01,8953.0,73.18,37.804286,8901.0,166.41,27.68,2015,7,-0.500000,-8.660254e-01,0,8302.0,8381.0,8361.0
1,2015-08-01,8535.0,72.58,63.944286,8531.0,167.12,27.85,2015,8,-0.866025,-5.000000e-01,0,8953.0,8302.0,8381.0
2,2015-09-01,8498.0,75.12,119.769714,8373.0,157.25,27.50,2015,9,-1.000000,-1.836970e-16,0,8535.0,8953.0,8302.0
3,2015-10-01,8330.0,78.86,153.807429,8324.0,143.69,26.60,2015,10,-0.866025,5.000000e-01,0,8498.0,8535.0,8953.0
4,2015-11-01,6511.0,87.29,316.494857,6508.0,92.62,25.04,2015,11,-0.500000,8.660254e-01,1,8330.0,8498.0,8535.0


In [21]:
# select the target
y = df["Electricity_Requirement"]

# We does require those columns

In [22]:
X = df.drop(
    columns=[
        "Date",
        "Electricity_Requirement",
        "Electricity_Supply",
        "Demand_Lag_1",
        "Demand_Lag_2",
        "Demand_Lag_3",
    ]
)

In [32]:
X.head()

,Humidity,Rainfall,Solar_Irradiance,Temperature,Year,Month,Month_sin,Month_cos,Festival
0,73.18,37.804286,166.41,27.68,2015,7,-0.500000,-8.660254e-01,0
1,72.58,63.944286,167.12,27.85,2015,8,-0.866025,-5.000000e-01,0
2,75.12,119.769714,157.25,27.50,2015,9,-1.000000,-1.836970e-16,0
3,78.86,153.807429,143.69,26.60,2015,10,-0.866025,5.000000e-01,0
4,87.29,316.494857,92.62,25.04,2015,11,-0.500000,8.660254e-01,1


In [23]:
print(X.columns)

Index(['Humidity', 'Rainfall', 'Solar_Irradiance', 'Temperature', 'Year',
       'Month', 'Month_sin', 'Month_cos', 'Festival'],
      dtype='object')


In [24]:
print("Features Shape :", X.shape)
print("Target Shape   :", y.shape)

Features Shape : (126, 9)
Target Shape   : (126,)


In [25]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [26]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (100, 9)
X_test  : (26, 9)
y_train : (100,)
y_test  : (26,)


In [27]:
X_train.head()

,Humidity,Rainfall,Solar_Irradiance,Temperature,Year,Month,Month_sin,Month_cos,Festival
0,73.18,37.804286,166.41,27.68,2015,7,-0.500000,-8.660254e-01,0
1,72.58,63.944286,167.12,27.85,2015,8,-0.866025,-5.000000e-01,0
2,75.12,119.769714,157.25,27.50,2015,9,-1.000000,-1.836970e-16,0
3,78.86,153.807429,143.69,26.60,2015,10,-0.866025,5.000000e-01,0
4,87.29,316.494857,92.62,25.04,2015,11,-0.500000,8.660254e-01,1


In [28]:
y_train.head()

0    8953.0
1    8535.0
2    8498.0
3    8330.0
4    6511.0
Name: Electricity_Requirement, dtype: float64

# saving the datasets

In [29]:
joblib.dump(X_train, "Demand_X_train.pkl")
joblib.dump(X_test, "Demand_X_test.pkl")

joblib.dump(y_train, "Demand_y_train.pkl")
joblib.dump(y_test, "Demand_y_test.pkl")

['Demand_y_test.pkl']

In [30]:
import os

for i in os.listdir():
    print(i)

00_Preprocessing(4).ipynb
00_Preprocessing.ipynb
01Randomforest.ipynb
02_XGBoost_Demand_Forecasting.ipynb
03_LightGBM_Demand_Forecasting.ipynb
Bidirectional_LSTM_Demand.ipynb
Demand_GRU_Baseline_08416.keras
Demand_LSTM_09803.keras
Demand_LSTM_Baseline.keras
Demand_LSTM_Best_08356.keras
Demand_X_Scaler.pkl
Demand_X_test.pkl
Demand_X_train.pkl
Demand_y_Scaler.pkl
Demand_y_test.pkl
Demand_y_train.pkl
GRU_Demand.ipynb
GRU_Demand_Forecasting_09944.keras
LSTM_Demand_Forecasting.ipynb
RandomForest_Demand_Forecasting_08610.pkl
Sarimax.ipynb
Simple_RNN.ipynb
